# Transit Capital Cost and Assested Values around Transit Stations 

PURPLE LINE CAPITAL COST AND VALUE RECAPTURE 

 the goal of this project is to create a way to take station locations geolocated , tax data by parcel which is geolocated , and station construciton cost. Then determine current use and under use of land around these stations. This is not a tool for determining if transit should be build into low value areas or determining the future change. It is a study into the intesnsity of land use through economic lens near transit stations. Taking studies from before as a way to predict value change and each properites share by land area and improvment values. 

planning implications  
    The main income source for a majority of local goverments in the United States is property tax. Even if a project that adds value to properites is not funded by property tax. Recovering that value in the form of property tax creates a more stable funding source for goverments. Another way to increse income is to maxiumize the income taken around already existing infrastructure. This study will take the improvment cost as an assumpiton that it measure the increasse use. Free riders and land value gain from infrastucture will also be shown in this study. Some landowners will recive a very large unearned increasse in land value at the cost of the tax payer.

# DATA 
I will be using the Maryland Transit Purple Line as an example in this code. This code and method should be repleblical for anyone with the follwoing data in a similar format 

TAX ASSESSMENT DATA WITH PARCEL MAP 

Maryland State has a centeral point to collect all tax data and parcels. This will be acessed in API form because the file size will be to large.
point data could also be used but I choose shape file because I wanted to create cloropath maps with parcel data. the data needs at least these features 
 
land area 
land value 
improvment value 
units 
improvement area 
homestead tax credit status 


Station Locations 

Stations names and locations in a csv or GEOJSON 

Per Station Costs

The easy way to get this value is take the cost of the whole project and divide it by the nubmer of stations. Some projects might also have bid documents that are publiclly available. The purple line has great diffrence in station constructuion costs per station. I did one example of the cost of constructuion then feed that method and the as builts to Chat gpt 5.4. I needed to coach it heavily including it making up stations. It gave me a csv with the station esitmates that I then spot checked with two other estimate costs mainly from the Federal Transit cost database as the basis for estimates. after correcting it and coaching it I was satsfitied with these estimates of construction cost. 

the CSV needs cost and station name 




In [1]:
# loading needed  packages 
import pandas as pd 
import geopandas as gpd 
import plotly.express as px
from ipyleaflet import GeoJSON, Map, CircleMarker, basemaps
import requests 
import json 
import yaml 
import os 

%load_ext autoreload
%autoreload 2 



# data loading 

# use this google drive link to download the data for the purple line anyalysis 
# https://drive.google.com/file/d/13IiTNyBXzwrRM8FBttzxJXgouoSF4a0G/view?usp=sharing 

# downlad the data from the folder and save it in the same directory as this notebook 
# own files can be placed in data folder with matching name 

In [2]:
# breaking loading data into two cells to becasue the smallparcles geojson file is very large 
# expect around 6 mins to load 
# small parcels is called that becasue it only contains the parcles in the two counites the purple line will go through. 
parcels = gpd.read_file('data/smallparcels.geojson')



In [3]:
# loading the rest of the data  
purplelinestations = gpd.read_file('data/purple_line_stations.geojson')

purplecost = pd.read_csv('data/purplelinestationestimates.csv') 

In [4]:
# changing FID in purplecost into int32

purplecost['FID'] = purplecost['FID'].astype('int32')

In [5]:
# renaming the columns in purplecost to match purplestations 
purplecost = purplecost.rename(columns={'Station': 'Name', '2022 Cost Estimate': 'Cost_Estimate_2022', '2024 Cost Estimate': 'Cost_Estimate_2024'})
# merging the two datafarmes on NAME , they are not exact same string they need to merge on the closet match 
purplestation = pd.merge(purplelinestations, purplecost, on='FID', how='left')
# making sure row count is the same as the original purplelinestations of 21 dispalying merge correct if it is 
purplestation.info()



<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 7 columns):
 #   Column                               Non-Null Count  Dtype   
---  ------                               --------------  -----   
 0   FID                                  21 non-null     int32   
 1   Name_x                               21 non-null     str     
 2   geometry                             21 non-null     geometry
 3   Name_y                               21 non-null     str     
 4   Estimated Low (USD millions)         21 non-null     int64   
 5   Estimated High (USD millions)        21 non-null     int64   
 6   Recommended Estimate (USD millions)  21 non-null     int64   
dtypes: geometry(1), int32(1), int64(3), str(2)
memory usage: 2.0 KB


In [6]:
# maping the geodataframe purplesation after the merge to make sure the geometry is still there and the merge did not mess up the geodataframe
# adding with basic base map to make sure the station are in the right place 
purplelinestationsonthemap = Map(center=(39.0, -77.0), zoom=10, basemap=basemaps.OpenStreetMap.Mapnik)
geo_json = GeoJSON(data=json.loads(purplestation.to_json()))    
purplelinestationsonthemap.add_layer(geo_json)
purplelinestationsonthemap




Map(center=[39.0, -77.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

# Creating Buffers around the stations 

In [7]:
# three buffers will be created around each purple line station 
# reasoning for this comes from this fredie mac study on home values and transit proximity in dc 
# https://www.freddiemac.com/fmac-resources/research/pdf/201909-Insight.pdf 
# buffers will be 0-1/4 mile, 1/4-1/2 mile, and 1/2-1 mile.
# Reproject to a projected CRS for accurate buffering in meters
projected_crs = 'EPSG:26985'  # NAD83 / Maryland
purplestation_proj = purplestation.to_crs(projected_crs)

# Convert miles to meters
mile_to_meter = 1609.34
quarter_mile_m = mile_to_meter / 4
half_mile_m = mile_to_meter / 2
one_mile_m = mile_to_meter

# Create buffers
stationbuffer1 = purplestation_proj.geometry.buffer(quarter_mile_m)
stationbuffer2 = purplestation_proj.geometry.buffer(half_mile_m)
stationbuffer3 = purplestation_proj.geometry.buffer(one_mile_m)

# Create ring buffers for 0-1/4 mile, 1/4-1/2 mile, 1/2-1 mile
ring_0to0_25 = stationbuffer1
ring_0_25to0_5 = stationbuffer2.difference(stationbuffer1)
ring_0_5to1 = stationbuffer3.difference(stationbuffer2)

# plotting the Rings on a map as a spot check 

In [8]:
# convert ring buffers back to geographic coordinates for mapping
ring_0to0_25_gdf = gpd.GeoDataFrame(geometry=ring_0to0_25, crs=projected_crs).to_crs(epsg=4326)
ring_0_25to0_5_gdf = gpd.GeoDataFrame(geometry=ring_0_25to0_5, crs=projected_crs).to_crs(epsg=4326)
ring_0_5to1_gdf = gpd.GeoDataFrame(geometry=ring_0_5to1, crs=projected_crs).to_crs(epsg=4326)

station_map = Map(center=(39.0, -77.0), zoom=10, basemap=basemaps.OpenStreetMap.Mapnik)

station_map.add_layer(
    GeoJSON(
        data=json.loads(ring_0to0_25_gdf.to_json()),
        style={'color': '#1f77b4', 'fillColor': '#1f77b4', 'fillOpacity': 0.15, 'weight': 1}
    )
)

station_map.add_layer(
    GeoJSON(
        data=json.loads(ring_0_25to0_5_gdf.to_json()),
        style={'color': '#ff7f0e', 'fillColor': '#ff7f0e', 'fillOpacity': 0.15, 'weight': 1}
    )
)

station_map.add_layer(
    GeoJSON(
        data=json.loads(ring_0_5to1_gdf.to_json()),
        style={'color': '#2ca02c', 'fillColor': '#2ca02c', 'fillOpacity': 0.12, 'weight': 1}
    )
)

station_map.add_layer(
    GeoJSON(
        data=json.loads(purplestation.to_json()),
        style={'color': '#000000', 'radius': 4, 'fillColor': '#000000', 'fillOpacity': 1}
    )
)

station_map

Map(center=[39.0, -77.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [9]:
# mapping intersecting of parcels with the buffers  

# its going to give a deprecaton warning but it still works when I tried to correct it wiht its sugestion it dosent work so im leaving it as is 

#reportjecting 

parcels_proj = parcels.to_crs(projected_crs)

parcelsin025 = parcels.loc[parcels_proj.intersects(stationbuffer1.unary_union)].copy()
parcelsn2505 = parcels.loc[parcels_proj.intersects(stationbuffer2.unary_union)].copy()
parcelsin051 = parcels.loc[parcels_proj.intersects(stationbuffer3.unary_union)].copy()

C:\Users\Harbaugh\AppData\Local\Temp\ipykernel_12116\3733411151.py:9: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  parcelsin025 = parcels.loc[parcels_proj.intersects(stationbuffer1.unary_union)].copy()
C:\Users\Harbaugh\AppData\Local\Temp\ipykernel_12116\3733411151.py:10: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  parcelsn2505 = parcels.loc[parcels_proj.intersects(stationbuffer2.unary_union)].copy()
C:\Users\Harbaugh\AppData\Local\Temp\ipykernel_12116\3733411151.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  parcelsin051 = parcels.loc[parcels_proj.intersects(stationbuffer3.unary_union)].copy()


# maping one of the parcel layers as a spot check 

In [1]:
parcelsin025projected = parcelsin025.to_crs(epsg=4326)

parcels_map = Map(center=(39.0, -77.0), zoom=10, basemap=basemaps.OpenStreetMap.Mapnik)
parcels_map.add_layer(
    GeoJSON(
        data=json.loads(parcelsin025projected.to_json()),
        style={'color': '#0072B2', 'fillColor': '#56B4E9', 'fillOpacity': 0.2, 'weight': 1}
    )
)

parcels_map

NameError: name 'parcelsin025' is not defined

# the data is now in small enough for gis  
# Below I will export all the buffers and parcels in layers as a geojson then as follows in the markdown folder the steps which where taken in gis 

In [11]:
# exporting the geojsons
parcelsin025.to_file("quatermileparcels.geojson", driver='GeoJSON')
parcelsn2505.to_file("quatertohalfmileparcels.geojson",driver='GeoJSON')
parcelsin051.to_file("halftowholeparcels.geojson",driver='GeoJSON')
ring_0_5to1.to_file("ringbufferhalfwhole.geojson",driver='GeoJSON')
ring_0_25to0_5.to_file("ringbufferquatertohalf.geojson",driver='GeoJSON')
ring_0to0_25 = stationbuffer1.to_file("ringbufferquater.geojson",driver='GeoJSON')
purplestation.to_file("stations.geojson",driver='GeoJSON')



 Steps in gis 
  json to feature class for all the exported GeoJsons
    making sure to keep them as point or polygon 
    I used model buidler here because im just running every function 3 times 

    1. remove all null and zero value parcels these are roads and public instutions NFMLNDVL is the features 
    2. erased parcels so that each of the 3 segments of parcels is unquie 
    3. spatial joined so that each parcel has a station affilated with it 
    4. each of the parcel groups has there added value calcuated and total value calcuated based off that fredie mac paper on dc metro and transit home price
    5. created three binary collums one for each group incase need layer 
    6. combine and sent back as geojson to come back into python to calcuate new collumns and create graphs (The final Layer will go back to gis to make better maps ) logical behind this is that gis feature calcuation is slow and cumbersome 

In [12]:
# importing new masterparcel geo_json

cleanparcels = gpd.read_file('data/masterparcel.geojson')



In [13]:
# making cleanparcels into a geodataframe 
cleanparceldf = gpd.GeoDataFrame(cleanparcels)

In [14]:
#purplecost = purplecost.rename(columns={'Station': 'Name', '2022 Cost Estimate': 'Cost_Estimate_2022', '2024 Cost Estimate': 'Cost_Estimate_2024'})
renameparcel = cleanparceldf.rename(columns={'NFMIMPVL' : 'Improvementvalue', 'NFMLNDVL': 'Landvalue', 'NFMTTLVL' : 'Fullvalue', 'LANDAREA' : 'Landsqf', 'SQFTSTRC' : 'Improvementsqf'})
# renaming cost columns 
renameparcel.columns.tolist()

['OBJECTID',
 'Join_Count',
 'TARGET_FID',
 'OBJECTID_1',
 'JURSCODE',
 'ACCTID',
 'CT2020',
 'BG2020',
 'GEOGCODE',
 'OOI',
 'RESITYP',
 'ADDRESS',
 'STRTNUM',
 'STRTDIR',
 'STRTNAM',
 'STRTTYP',
 'STRTSFX',
 'STRTUNT',
 'ADDRTYP',
 'CITY',
 'ZIPCODE',
 'OWNNAME1',
 'OWNNAME2',
 'NAMEKEY',
 'OWNADD1',
 'OWNADD2',
 'OWNCITY',
 'OWNSTATE',
 'OWNERZIP',
 'OWNZIP2',
 'PREMSNUM',
 'PREMSDIR',
 'PREMSNAM',
 'PREMSTYP',
 'PREMCITY',
 'PREMZIP',
 'PREMZIP2',
 'LEGAL1',
 'LEGAL2',
 'LEGAL3',
 'DR1CLERK',
 'DR1LIBER',
 'DR1FOLIO',
 'TOWNCODE',
 'DESCTOWN',
 'SUBDIVSN',
 'DSUBCODE',
 'DESCSUBD',
 'PLAT',
 'PLTLIBER',
 'PLTFOLIO',
 'SECTION',
 'BLOCK',
 'LOT',
 'MAP',
 'GRID',
 'PARCEL',
 'ZONING',
 'ZNCHGDAT',
 'RZREALDAT',
 'CIUSE',
 'DESCCIUSE',
 'EXCLASS',
 'DESCEXCL',
 'LU',
 'DESCLU',
 'ACRES',
 'Landsqf',
 'LUOM',
 'WIDTH',
 'DEPTH',
 'PFUW',
 'PFUS',
 'PFLW',
 'PFSP',
 'PFSU',
 'PFIC',
 'PFIH',
 'RECIND',
 'PERMITTYP',
 'YEARBLT',
 'Improvementsqf',
 'STRUGRAD',
 'DESCGRAD',
 'STRUCNST',


In [32]:
# creating new columns for analysis  
#df['new_column'] = df['column_A'] + df['column_B'] 

# converting lot size to sqf 

renameparcel['lotsqf'] = renameparcel['ACRES'] * 43560

# value per sqf of land 
renameparcel['Valuegainedpersqf'] = renameparcel['Valueadded'] / renameparcel['lotsqf']

# value per sqf of improvemnt 
renameparcel['Valuegainedperimprovementsqf'] = renameparcel['Valueadded'] / renameparcel['Improvementsqf']

# Floor area ratio 

renameparcel['FAR'] = renameparcel['Improvementsqf'] / renameparcel['lotsqf']


# value gained / (units)
 
renameparcel['Valuegainedunit'] = renameparcel['Valueadded'] / renameparcel['BLDG_UNITS']

# value gained per unit normalized per acer 

renameparcel['Unitsperacre'] = renameparcel['BLDG_UNITS'] / renameparcel['ACRES']

renameparcel['Valueperunitsperacre'] = renameparcel['Valueadded'] / renameparcel['Unitsperacre']

# create estimated md taxes off new value 
# MD tax rate avearge is 0.009 

renameparcel['expectedtaxes'] = renameparcel['Adjustedvalue'] * 0.009 
# creating orginal taxes

renameparcel['ogexpectedtaxes'] = renameparcel['Fullvalue'] *0.009


In [33]:
# removing everything that is DESCLU exmept becasue they dont pay taxes  
#df = df[df['column_name'] != 'value_to_remove'] 
# remvoing comerical exmept 

removedparcel = renameparcel[renameparcel['DESCLU'] != 'Exempt']

finalparceldf = removedparcel[removedparcel['DESCLU'] != 'Exempt Commercial']

In [34]:
#how many parcels are in each of the distance groups 

parcelbygroup = finalparceldf.groupby('Name_x')['OBJECTID_1'].count()

parcelbygroupdf = parcelbygroup.to_frame()
parcelbygroupdf = parcelbygroupdf.rename(columns={'OBJECTID_1' : 'station count'})
#df.to_csv('filename.csv', index=False)
# making it a csv
parcelbygroupdf.to_csv('Numberofparcels.csv', index=False) 




In [40]:
# total taxes added up per station and station cost 
taxincomebystation = finalparceldf.groupby('Name_x')['expectedtaxes'].sum()
taxincomebystationdf = taxincomebystation.to_frame()
taxincomebystationdf['Expected Taxes In Millions'] = taxincomebystationdf['expectedtaxes']/ 1000000

# join this back to the orginal purple line station estimate 
jointaxincome = pd.merge(taxincomebystationdf , purplestation, on='Name_x', how='left',)


# making this a csv
jointaxincome.to_csv('taxincomebystation.csv' , index=False)

In [41]:
# everything is now finalized and ready for gis to make maps and the tax income csv is the answer 
# exporting finalparceldf as a geojson 
# if this throws and errror run it again 
# works the second time 
finalparceldf.to_file("finalparcellayer.geojson",driver='GeoJSON')